# Solutions · Chapter 02-01 · What is a row?

Worked answers with reasoning. E4 and E14 are the two that pay for the chapter: one shows the
mean-of-means gap closing exactly when the groups become equal, the other shows the wrong grain
producing a perfect score and a useless model.

Self-contained: run from the top with a fresh kernel.

In [ ]:
import numpy as np
import pandas as pd

events = pd.DataFrame({
    "rental_id": range(1, 13),
    "station": ["A"] * 10 + ["B"] * 2,
    "day": [1, 1, 1, 2, 2, 2, 2, 3, 3, 3, 1, 2],
    "customer": ["c1", "c1", "c2", "c1", "c1", "c2", "c3", "c1", "c1", "c4", "c2", "c5"],
    "duration_min": [10, 10, 12, 10, 10, 12, 90, 10, 10, 8, 12, 15],
})
per_station_day = (events.groupby(["station", "day"])
                   .agg(rentals=("rental_id", "size")).reset_index())
per_customer = (events.groupby("customer")
                .agg(rentals=("rental_id", "size"),
                     mean_duration=("duration_min", "mean")).reset_index())
print(len(events), "events |", len(per_station_day), "station-days |", len(per_customer), "customers")

## E1 · Definitions

**The unit of observation** is what one row represents - the thing you are making a statement about.
It is a decision you make to match the decision you are supporting, not a property the file hands
you.

**The four kinds of column:**

- **Target** - what you want to predict, defined at the grain of the row.
- **Feature** - information you will genuinely have at the moment of prediction.
- **Identifier** - names the row and carries no information about the outcome.
- **Metadata** - describes the record rather than the thing recorded (export time, source file,
  ingestion batch).

## E2 · Identifier versus derived feature

`customer` is a **label for a person**. The string `"c1"` says nothing about how long that person
rents for; it is only useful because *other rows share it*. Handed to a model as a category, it lets
the model memorise individuals - which works perfectly for customers seen in training and not at all
for anyone new.

**Legitimate features derived from it**, all with the same critical qualifier:

- number of rentals **before this one**
- average duration **so far**
- days since this customer's **previous** rental
- whether this is their first rental

The qualifier is the whole thing. `rentals_so_far` computed from the customer's *entire* history
includes rentals that happen after the row you are predicting - information from the future, which
is target leakage (04-05). The identifier is the **key you group by** to compute history, never the
input itself.

**And the deeper reason this matters:** a model that memorises `c1` cannot help with `c9`, and every
production system is mostly customers it has never seen.

## E3 · The mean-of-means rule

> The average of group averages is not the average of the data, **unless every group has the same
> number of rows.**

When group sizes differ, the overall average weights each row equally, so bigger groups pull it
further; the average of averages weights each *group* equally, so a group with two rows counts as
much as one with two hundred.

Neither is wrong. They answer different questions - "what is a typical row?" versus "what is a
typical group?" - and E4 shows the gap closing exactly when the sizes are equalised.

## E4 · Watching the gap close

In [ ]:
def both_averages(table, label):
    overall = table["rentals"].mean()
    of_means = table.groupby("station")["rentals"].mean().mean()
    sizes = table["station"].value_counts().sort_index().to_dict()
    print(f"{label}\n  group sizes {sizes}"
          f"\n  per station-day {overall:.4f} | mean of station means {of_means:.4f}"
          f" | gap {overall - of_means:+.4f}")

both_averages(per_station_day, "as recorded (A open 3 days, B open 2)")

extended = pd.concat([per_station_day,
                      pd.DataFrame([{"station": "B", "day": 3, "rentals": 1}])], ignore_index=True)
both_averages(extended, "\nwith B also open on day 3 (both open 3 days)")

**As recorded:** 2.4000 against 2.1667, a gap of 0.2333.
**With equal group sizes:** 2.1667 against 2.1667 - the gap is exactly zero.

**Which number moved?** The **per-station-day** average fell from 2.400 to 2.167. The mean of
station means did not move at all: A's average is still 10/3 and B's is still 1.0, so their mean is
unchanged.

That is the mechanism made visible. Adding a quiet day to the smaller station changed the *weights*
in the overall average - B now contributes three of six days instead of two of five - while leaving
the per-station picture untouched.

**The transferable point:** the average of averages is *insensitive to group size by design*. That
is exactly why you would choose it - "how does a typical station perform, regardless of how long it
was open?" - and exactly why it is the wrong answer to "how busy is a typical day?".

## E5 · Three numbers from twelve rentals

In [ ]:
durations = sorted(events["duration_min"])
print("sorted durations:", durations)
print(f"mean per rental  : 209 / 12 = {events['duration_min'].mean():.2f} min")
print(f"median per rental: (10 + 10) / 2 = {events['duration_min'].median():.2f} min")
print(f"mean per customer: 135 /  5 = {per_customer['mean_duration'].mean():.2f} min")

| Number | Value | Why it differs |
|---|---|---|
| Mean per rental | **17.42** | Every rental counts once, so c1's six short trips carry half the weight - but c3's single 90-minute trip still drags it up by about 6 minutes on its own |
| Median per rental | **10.00** | The middle of twelve sorted values, both of which are 10. Completely unmoved by the 90 |
| Mean per customer | **27.00** | Every *person* counts once, so c3's 90 minutes counts as much as c1's entire six-rental history |

**The three together tell a story none of them tells alone:** most rentals are about 10 minutes,
one is nine times that, and how much that outlier matters depends entirely on whether you are
counting trips or people.

**What I would actually report:** "the median rental is 10 minutes; one of twelve ran to 90". That
is two numbers, both honest, and it prevents every downstream misunderstanding. A single mean of
17.42 describes no rental that happened.

## E6 · One row per customer-day

In [ ]:
per_customer_day = (events.groupby(["customer", "day"])
                    .agg(rentals=("rental_id", "size"),
                         total_minutes=("duration_min", "sum")).reset_index())
print(f"{len(per_customer_day)} rows")
print(per_customer_day.to_string(index=False))

**Eight rows.** Fewer than twelve, because some customers rented more than once on the same day
(c1 twice on day 1, twice on day 2, twice on day 3); more than five, because customers appear on
several days.

**What it can answer that none of the other four can:** *how does one person's behaviour change from
day to day?* For example - do customers who rent twice in a day take shorter trips than customers
who rent once? Is a customer's second day heavier than their first? Does a heavy day predict a
return the next day?

Those are all questions about **a person within a period**, and they need exactly this grain. The
per-customer table has collapsed the days away; the per-rental table has no notion of a person's day
as a unit.

**And note the modelling consequence:** this is the grain you want for churn or engagement
prediction with time - one row per customer per period, target defined on the *next* period. That
shape is what module 09 calls a panel, and it is where most real customer models live.

## E7 · Describing a grain

In [ ]:
def describe_grain(df, name):
    unique_cols = [c for c in df.columns if df[c].is_unique]
    print(f"{name:<18} {len(df):>3} rows | uniquely identifying columns: {unique_cols or 'none singly'}")

per_rental, per_station = events, (events.groupby("station")
                                   .agg(rentals=("rental_id", "size"),
                                        days_open=("day", "nunique")).reset_index())
for nm, tb in [("per rental", per_rental), ("per station-day", per_station_day),
               ("per station", per_station), ("per customer", per_customer)]:
    describe_grain(tb, nm)

**What it reveals about `per_station`:** it has **two rows**, and `station`, `rentals` and
`days_open` are *all* unique - because with two rows, almost any column is unique by accident.

That is the finding worth having: **a table this small cannot support a model, and it cannot support
a comparison either.** Any difference between two stations is a difference between two numbers, with
no way to tell signal from noise. If someone asks "is station A better than station B?", the honest
answer at this grain is "we have one observation of each".

**The general lesson about uniqueness checks:** `is_unique` finding a candidate key is useful on a
large table and meaningless on a tiny one. The same diagnostic changes value with the row count -
which is itself a good reason to always print the row count first.

Note also that no *single* column identifies a station-day: it takes `station` **and** `day`
together. Composite keys are the normal case, and a table whose grain needs two columns to express
is one where a join is easy to get wrong (01-05).

## E8 · "Our average customer rents for 27 minutes"

Three questions:

1. **What is the row - a customer or a rental?** 27 minutes is the per-customer figure. The
   per-rental figure is 17.42. For a *pricing* decision the relevant unit is almost always the
   rental, because that is what gets charged. Quoting the customer figure in a pricing model
   overstates the average paid trip by more than half.
2. **Is the mean the right summary?** The median rental is 10 minutes. The mean is pulled up by one
   90-minute trip out of twelve. Pricing built on a mean that no actual trip resembles will
   mis-forecast revenue for the great majority of transactions.
3. **Over what period, and which customers?** Twelve rentals from five customers across three days
   is not a description of "our customers". Two of the five appear exactly once. Any statement about
   typical behaviour needs enough history per person to be a description rather than a coincidence.

**And a fourth, which is really the first:** *what decision does this number feed?* If it is pricing
per minute, you need the distribution of trip durations, not any single average - because revenue
depends on the whole shape, and the long trips are where the money and the fleet pressure both are.

## E9 · The churn model at the wrong grain

**In order of severity:**

1. **The grain does not match the target.** Churn is a property of a *customer*; the table has one
   row per *rental*. A customer with 200 rentals contributes 200 identical labels and a customer with
   1 contributes one, so the model is trained mostly on heavy users - and heavy users are precisely
   the ones who do not churn. The dataset silently weights itself against the thing being predicted.
2. **The split leaks.** With random splitting, the same customer's rentals land on both sides. The
   model sees "customer 4471 churned" in training and is tested on another of customer 4471's rows,
   which carries the same label. **This is what explains the 0.96**: the model is not predicting
   churn, it is recognising customers. Group-aware splitting (04-04) is the fix.
3. **Rental-level features cannot describe a customer.** One rental's duration says little about
   whether a person will leave. The informative features - trend in frequency, time since last
   rental, change in behaviour - only exist once you aggregate to the customer.

**The fix is not a better model.** Rebuild the table at one row per customer, define the target with
an explicit observation window and prediction horizon ("churned = no rental in the 60 days after the
cut-off"), build features only from *before* the cut-off, and split by customer. That is chapters
04-01 and 04-04, and this exercise is why they exist.

**The tell you can use immediately:** an accuracy far above what the problem plausibly allows, on a
table where one entity owns many rows. E14 reproduces it deliberately.

## E10 · "What is the first question you ask?"

> What does one row represent? I want it in a sentence - "one row is one station on one day" - and I
> want it written at the top of the notebook, because everything else depends on it: what the target
> means, which features can exist, whether an average is weighted the way anyone intends, and how the
> data must be split. Most datasets arrive at the grain that was convenient for whoever exported
> them, which is rarely the grain of the decision being supported, so the second question is usually
> whether I need to aggregate. When the answer is unclear, I do not guess - I ask what decision this
> is for, because the thing being decided about *is* the row. If nobody can tell me, that is the
> finding: the problem is not yet defined well enough to model, and discovering that on day one is
> much cheaper than discovering it after a month.

**What is being tested:** whether you treat framing as work. Candidates who answer "check for
missing values" have named a task; this answer names a decision.

## E11 · Predicting stock-outs

**One row =** one product at one store for one week. (Product alone loses the store; store alone
loses the product; without the week there is no "next week".)

**Target:** did this product run out of stock in this store during this week - a yes/no. Or, if the
decision is about severity, hours out of stock, measured in hours.

**Two features available at prediction time:** units sold in the previous four weeks at this
store-product; current stock on hand at the moment the prediction is made. Both are known before the
week begins.

**A column that looks like a feature and is not:** *units sold during the week being predicted.* It
is in every historical row, it is enormously predictive, and it does not exist when you need it -
and worse, a stock-out *caps* it, so it partly encodes the answer. Also disqualified: deliveries
received during the week, and end-of-week stock level.

**The subtlety worth stating:** stock-outs are censored. A product that sold out on Tuesday shows
low sales for the week, so "low sales" and "ran out" are entangled in the history. Training a model
on sales without accounting for that teaches it that low demand causes stock-outs, which is
backwards. That is a data-generating-process problem, and it is 02-02's subject.

## E12 · The school's exam data

| Question | One row is | Target | The difficulty |
|---|---|---|---|
| (a) Which students need extra support? | one **student** (perhaps per term) | a risk score or a flag, defined on future performance | Requires a cut-off in time: features from before it, outcome after it. Otherwise you "predict" marks you already used |
| (b) Which subjects are getting harder? | one **subject-year** (or subject-cohort) | mean mark, or pass rate | The cohort changes every year, so a falling mark may mean the subject got harder *or* that different students took it. This is confounding (00-04), not a data problem |
| (c) Will this student pass this exam? | one **student-exam sitting** | pass/fail | Sittings by the same student are not independent; a random split puts a student on both sides, so the split must be by student (04-04) |

**The instructive part is that all three come from one file** with the same four columns. The file
does not change; the row does, and with it the target, the split and the difficulty.

**And (b) is the one to be most careful about**, because it is the only one that is not really a
prediction question at all. "Is the subject getting harder" asks about a cause, and comparing
different cohorts across years is exactly the non-comparable-groups problem from 00-04.

## E13 · Explaining it to Maria

> You counted days and your assistant counted stations. Station A was open three days and B only
> two, so when you average the days, A counts more - and A is the busier one. When you average the
> two stations instead, each counts once, however long it was open. Both sums are right; they answer
> slightly different questions.

(59 words.)

**Where it stops being just an arithmetic point:** which one Maria wants depends on what she is
about to do. Deciding how many bikes to load per morning is a question about days. Deciding whether
to keep station B open is a question about stations. The arithmetic cannot tell her which; only the
decision can.

## E14 · The wrong grain, and a perfect score

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split

# SYNTHETIC: 60 customers, 3-20 rentals each, churn assigned per customer AT RANDOM.
rng = np.random.default_rng(3)
names = [f"c{i:02d}" for i in range(60)]
counts = rng.integers(3, 21, 60)
churn_of = dict(zip(names, rng.random(60) < 0.5))

rows = pd.DataFrame({"customer": np.repeat(names, counts)})
rows["churned"] = rows["customer"].map(churn_of)

X = pd.get_dummies(rows["customer"])          # the identifier, one-hot encoded
y = rows["churned"]
print(f"{len(rows)} rentals from {len(names)} customers | churn share {y.mean():.2f}")

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)
random_acc = accuracy_score(y_te, LogisticRegression(max_iter=2000).fit(X_tr, y_tr).predict(X_te))

train_idx, test_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=0)
                           .split(X, y, groups=rows["customer"]))
grouped_acc = accuracy_score(y.iloc[test_idx],
                             LogisticRegression(max_iter=2000)
                             .fit(X.iloc[train_idx], y.iloc[train_idx]).predict(X.iloc[test_idx]))
baseline = max(y.iloc[test_idx].mean(), 1 - y.iloc[test_idx].mean())

print(f"random split  (customers on both sides): accuracy {random_acc:.3f}")
print(f"grouped split (test customers unseen)  : accuracy {grouped_acc:.3f}")
print(f"majority-class baseline on that test set: {baseline:.3f}")

**A perfect 1.000 with a random split. 0.243 once each customer is kept entirely on one side -
against a majority-class baseline of 0.757.**

Read those three numbers again. The random split reports a flawless model. The honest split reports
a model that is *worse than refusing to think*.

**The mechanism.** The only feature is the customer identifier, one-hot encoded, and the label was
assigned at random per customer - so there is genuinely nothing to learn. Under a random split every
customer in the test set also appears in the training set, so the model has already been told that
customer's label and simply looks it up. It is not predicting; it is 00-01's rule C, the memoriser,
in a different costume.

Under a grouped split the held-out customers were never seen. All their one-hot columns are zero, so
the model has no information at all, and whichever way its intercept happens to fall decides every
prediction - here, the wrong way for this particular set of held-out customers. That is why it lands
*below* the baseline rather than on it: with nothing to go on, a model does not gracefully degrade to
the sensible answer, it produces whatever its fitted constant says.

**This is why the grain and the split are one decision, not two.** With one row per rental and a
target defined per customer, a random split *cannot* be honest - the duplication of the label across
a customer's rows guarantees leakage. Fixing it needs either a grouped split or a rebuilt table at
the customer grain, and usually both.

**What to take away for real work:** if one entity owns many rows and the target is a property of
that entity, assume leakage until you have proved otherwise. The symptom is a score that is too good;
the cause is almost never the model. Chapter 04-04 builds the splitters and 04-05 catalogues the
other three ways this happens.

---

## Where to go next

Back to the chapter for the mastery check and flashcards, then **02-02 · Where data comes from:
provenance and the collection process**.